In [ ]:
from pyspark import *
from pyspark import SparkConf, SparkContext
from pyspark.sql import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *


# Databricks supplies spark; create a local session only outside Databricks.
if "spark" not in globals():
    spark = SparkSession.builder.master("local[2]").appName("Scenerio30").getOrCreate()


In [ ]:
data1 = [
    (1, "A", "A", 1000000),
    (2, "B", "A", 2500000),
    (3, "C", "G", 500000),
    (4, "D", "G", 800000),
    (5, "E", "W", 9000000),
    (6, "F", "W", 2000000),
]
df1 = spark.createDataFrame(data1, ["emp_id", "name", "dept_id", "salary"])
df1.show()

data2 = [("A", "AZURE"), ("G", "GCP"), ("W", "AWS")]
df2 = spark.createDataFrame(data2, ["dept_id1", "dept_name"])
df2.show()

In [ ]:
joindf = df1.join(df2, df1["dept_id"] == df2["dept_id1"], "inner").drop("dept_id1")
joindf.show()

In [ ]:
wn = Window.partitionBy("dept_id").orderBy(col("salary").desc())

rankdf = joindf.withColumn("rank", dense_rank().over(wn))
rankdf.show()

In [ ]:
finaldf = (
    rankdf.filter(col("rank") == 2)
    .drop("rank")
    .select("emp_id", "name", "dept_name", "salary")
)
finaldf.show()